In [1]:
from gitsource import GithubRepositoryDataReader
from minsearch import Index


def load_lessons():
    reader = GithubRepositoryDataReader(
        repo_owner="DataTalksClub",
        repo_name="llm-zoomcamp",
        commit_id="8c1834d",
        allowed_extensions={"md"},
        filename_filter=lambda path: "/lessons/" in path,
    )

    lessons = reader.read()
    return lessons


def format_lessons(lessons):
    documents = []

    for lesson in lessons:
        doc = lesson.parse()
        documents.append(doc)
    return documents
    
    
def build_index(documents):
    index = Index(
        text_fields = ['content'],
        keyword_fields = ['filename']
    )
    index.fit(documents)
    return index

In [2]:
lessons = load_lessons()
documents = format_lessons(lessons)
index = build_index(documents)

len(documents)

72

## Q1 answer: 72

In [3]:
from minsearch import Index
def build_index(documents):
    index = Index(
        text_fields = ['content'],
        keyword_fields = ['filename']
    )
    index.fit(documents)
    return index

In [4]:
index = build_index(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)
print(results)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [5]:
results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

## Q2 answer: 01-agentic-rag/lessons/14-agentic-loop.md

In [31]:
from groq import Groq
from dotenv import load_dotenv
load_dotenv()
client = Groq()

INSTRUCTIONS = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

PROMPT_TEMPLATE = """
Use the following lesson content to answer the student's question.

Context:
{context}

Question:
{question}
""".strip()

class RAGBase:
    def __init__(
        self, 
        index, 
        client=client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model="llama-3.3-70b-versatile"
        ):
        self.index = index
        self.client = client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model
    
    def search_db(self, question):
        return self.index.search(
            question,
            num_results=3
            )
        
    def format_context(self, documents):
        context = []
        for doc in documents:
            context.append(f"""
            file name: {doc['filename']}
            Content: {doc['content']}
            """
            )
        return "\n\n".join(context)
        
    def build_prompt(self, question, context):
        return self.prompt_template.format(question=question, context=context)
    
    def llm(self, prompt):
        model = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", 
                "content": self.instructions},
                {"role": "user", "content": prompt},
            ],
        )
        return model.choices[0].message.content, model.usage.total_tokens

    def rag(self, question):
        docs = self.search_db(question)
        context = self.format_context(docs)
        prompt = self.build_prompt(question, context)
        answer, usage = self.llm(prompt)
        return answer, usage
    
    



In [32]:
rag = RAGBase(index)
answer, token_usage = rag.rag("How does the agentic loop keep calling the model until it stops? ")
token_usage

6187

## Q3 answer: 7000

In [33]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [34]:
len(chunks)

295

## Q4 answer: 295

In [35]:
index_chunks = build_index(chunks)
rag_chunks = RAGBase(index_chunks)
answer, token_usage_chunks = rag_chunks.rag("How does the agentic loop keep calling the model until it stops? ")
token_usage_chunks

1706

In [36]:
token_usage/token_usage_chunks

3.626611957796014

## Q5 answer: 3× fewer

In [37]:
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [52]:
def search(query: str) -> list[dict]:
    """
    Search the lessons database for chunks matching the given query.
    """
    return index.search(
        query,
        num_results=3
    )


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lessons database for chunks matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course lesson content."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}
    
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [53]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons database for chunks matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course lesson content.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [54]:
import os
from openai import OpenAI
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

groq_client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

llm_client = OpenAIChatCompletionsClient(
    model="llama-3.1-8b-instant",
    client=groq_client
)


In [55]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [56]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


/workspaces/llm-zoomcamp-code/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:542: UnknownModelWarning: No pricing data for model 'llama-3.1-8b-instant'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(
